In [15]:
%pip install torch torchvision matplotlib numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import Subset
import matplotlib.pyplot as plt

# Set a fixed seed for everyone on the team to use!
np.random.seed(42)
torch.manual_seed(42)

In [22]:
# Run this in a separate cell to download the dataset into Colab
!wget https://raw.githubusercontent.com/jcpeterson/cifar-10h/master/data/cifar10h-probs.npy

zsh:1: command not found: wget


In [27]:
# 1. Download the standard CIFAR-10 Test Set (the images)
# Note: We use train=False because CIFAR-10H labels match the test batch.

# Try using requests library if available, otherwise fallback to torch download
try:
    import requests
    requests.packages.urllib3.disable_warnings()
    
    # Check if dataset already exists
    if not os.path.exists('./data/cifar-10-batches-py'):
        print("Downloading CIFAR-10...")
        url = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
        response = requests.get(url, verify=False)
        os.makedirs('./data', exist_ok=True)
        
        with open('./data/cifar-10-python.tar.gz', 'wb') as f:
            f.write(response.content)
        
        import tarfile
        with tarfile.open('./data/cifar-10-python.tar.gz', 'r:gz') as tar:
            tar.extractall('./data/')
        print("Download and extraction complete!")
    
    full_cifar10_test = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=False, transform=transforms.ToTensor()
    )
except:
    # Fallback: try direct download
    print("Using torchvision download...")
    full_cifar10_test = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transforms.ToTensor()
    )

# 2. Load the CIFAR-10H Soft Labels (the "human cloud" of opinions)
soft_labels = np.load('cifar10h-probs.npy')


Download and extraction complete!


FileNotFoundError: [Errno 2] No such file or directory: 'cifar10h-probs.npy'

In [ ]:
# Check 1: Do we have the same number of images and labels?
print(f"Images: {len(full_cifar10_test)}, Labels: {soft_labels.shape[0]}") # Should both be 10,000

# Check 2: Do the soft labels sum to 1.0? [cite: 66]
sums = np.sum(soft_labels, axis=1)
if np.allclose(sums, 1.0):
    print("Sanity Check Passed: All distributions sum to 1.")
else:
    print("Warning: Distributions do not sum to 1!")


NameError: name 'full_cifar10_test' is not defined

In [18]:
# Create a list of indices from 0 to 9999
indices = list(range(10000))
np.random.shuffle(indices)

# Define the split points
train_idx = indices[:6000]
val_idx = indices[6000:8000]
test_idx = indices[8000:]

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

Train: 6000, Val: 2000, Test: 2000
